In [0]:
"""
Data Quality & Reconciliation Test Suite for Gold Medallion Layer.
Framework: pytest + PySpark

This module validates the integrity and completeness of Gold layer tables
by checking for non-empty tables, unique primary keys, referential integrity,
and row-count reconciliation between Silver and Gold layers.
"""

# ── Dependencies ────────────────────────────────────────────────────────────
import pytest                         # Test framework
from pyspark.sql import SparkSession  # Spark entry-point

# ── Shared Fixtures ──────────────────────────────────────────────────────────
# A single SparkSession is created once per test session and reused across all
# tests to avoid the overhead of repeatedly starting/stopping Spark.
@pytest.fixture(scope="session")
def spark():
    return SparkSession.builder.getOrCreate()

# ── Test 1 : Completeness ────────────────────────────────────────────────────
# Validates that the Gold fact table is not empty after the pipeline has run.
# A zero-row count indicates a failed or missing ingestion job.
def test_gold_fact_table_not_empty(spark):
    """Ensure fact_wikipedia_edits has ingested records."""
    count = spark.table("dbr_dev.wikimediademo_gold.fact_wikipedia_edits").count()
    assert count > 0, "Assertion Failed: fact_wikipedia_edits table is empty."

# ── Test 2 : Primary Key Uniqueness ──────────────────────────────────────────
# Groups dim_project by its surrogate key and looks for any key that appears
# more than once. Duplicate PKs would break JOIN correctness in downstream
# reports and BI tools.
def test_dim_project_primary_keys_unique(spark):
    """Ensure surrogate primary keys in dim_project are strictly unique."""
    duplicates = (
        spark.table("dbr_dev.wikimediademo_gold.dim_project")
        .groupBy("project_key")   # Collapse to one row per key value
        .count()                   # Count occurrences per key
        .filter("count > 1")       # Keep only the duplicate keys
        .count()                   # Total number of offending keys
    )
    assert duplicates == 0, f"Assertion Failed: Found {duplicates} duplicate keys in dim_project."

# ── Test 3 : Referential Integrity ───────────────────────────────────────────
# Performs a LEFT JOIN from the fact table to the dimension table.
# Any fact row whose project_key has no match in dim_project surfaces as NULL
# on the right side — these "orphan" records produce incorrect aggregations
# or missing labels in reports.
def test_referential_integrity_fact_to_dim_project(spark):
    """Ensure all facts map to a valid dimension key (No Orphan Records)."""
    orphan_count = spark.sql("""
        SELECT count(*) AS orphan_count
        FROM dbr_dev.wikimediademo_gold.fact_wikipedia_edits f
        LEFT JOIN dbr_dev.wikimediademo_gold.dim_project p 
            ON f.project_key = p.project_key
        WHERE p.project_key IS NULL  -- NULL means no matching dimension row found
    """).collect()[0]["orphan_count"]
    
    assert orphan_count == 0, f"Assertion Failed: Found {orphan_count} orphan fact records without dimension mappings."

# ── Test 4 : Silver → Gold Row-Count Reconciliation ──────────────────────────
# Compares the total number of rows in the Silver source against the Gold fact
# table. A mismatch means records were dropped or duplicated during the
# Silver → Gold transformation, which breaks data lineage guarantees.
def test_silver_to_gold_reconciliation(spark):
    """Ensure 100% reconciliation matching between Silver stream source and Gold fact table."""
    # Change wikipedia_edits -> silver_wikipedia_edits
    silver_count = spark.table("dbr_dev.wikimediademo_silver.silver_wikipedia_edits").count()
    gold_count = spark.table("dbr_dev.wikimediademo_gold.fact_wikipedia_edits").count()
    
    assert silver_count == gold_count, (
        f"Reconciliation Mismatch: Silver count ({silver_count}) != Gold fact count ({gold_count})"
    )

In [0]:
from pyspark.sql import SparkSession

active_spark = SparkSession.builder.getOrCreate()

tests = [
    test_gold_fact_table_not_empty,
    test_dim_project_primary_keys_unique,
    test_referential_integrity_fact_to_dim_project,
    test_silver_to_gold_reconciliation
]

print("=== Running Gold Layer Quality Tests ===")
for test in tests:
    try:
        test(active_spark)
        print(f"PASSED: {test.__name__}")
    except AssertionError as e:
        print(f"FAILED: {test.__name__} -> {e}")
    except Exception as e:
        print(f"ERROR:  {test.__name__} -> {e}")
print("========================================")